# Dr Plant — Lemon V3 exploratory training

Open this notebook in Google Colab from your Android browser. The dataset archive is kept in Google Drive so you only need to obtain it once. On later runs the notebook reuses the saved ZIP and skips the Mendeley API entirely.

**Important:** this is an initial within-dataset exploratory baseline only. The published record does not provide per-image site/plant/session IDs, so the split is not independent. Results must not be represented as accuracy on unseen Syrian orchards.

In [ ]:
# Select Runtime > Change runtime type > GPU (T4 if available) before running.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
WORK = Path('/content/drplant-work')
DRIVE = Path('/content/drive/MyDrive/DrPlant')
WORK.mkdir(parents=True, exist_ok=True)
DRIVE.mkdir(parents=True, exist_ok=True)
print('GPU:', __import__('torch').cuda.is_available())

In [ ]:
# Clone the project reliably, including when this cell is re-run.
import os, subprocess
from pathlib import Path

REPO = Path('/content/drplant-source')
if (REPO / '.git').exists():
    print('Repository already exists; updating it...')
    update = subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], text=True, capture_output=True)
    if update.returncode != 0:
        raise RuntimeError('Could not update the existing Dr Plant checkout:\\n' + update.stderr)
else:
    if REPO.exists():
        raise RuntimeError(f'{REPO} exists but is not a Git repository. In Colab, delete that folder from the Files panel or restart the runtime, then rerun this cell.')
    print('Downloading Dr Plant from GitHub...')
    clone = subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/jonbander/dr-plant.git', str(REPO)], text=True, capture_output=True)
    if clone.returncode != 0:
        raise RuntimeError('GitHub download failed. Check that Colab internet access is available, then rerun this cell. Details:\\n' + clone.stderr)

os.chdir(REPO)
required = [Path('scripts/audit_image_classification_dataset.py'), Path('scripts/prepare_exploratory_splits.py'), Path('scripts/train_image_classifier.py')]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError('The downloaded repository is missing required training scripts: ' + ', '.join(missing))

print('Project folder:', Path.cwd())
print('Required training scripts: OK')
subprocess.run([__import__('sys').executable, '-m', 'pip', 'install', '-q', 'Pillow', 'scikit-learn'], check=True)


In [ ]:
# Reuse the saved archive first. Only try the Mendeley API when no valid ZIP is already in Drive.
import hashlib, json, requests, time, zipfile, shutil
from pathlib import Path

DATASET_ID = '8d9fv6kpt3'
VERSION = 3
ARCHIVE = DRIVE / '8d9fv6kpt3-v3.zip'
PARTIAL = DRIVE / '8d9fv6kpt3-v3.zip.part'

def check_zip(path):
    if not path.is_file() or path.stat().st_size < 1024 * 1024:
        return False, 'file missing or unexpectedly small'
    if not zipfile.is_zipfile(path):
        return False, 'file is not a valid ZIP archive'
    try:
        with zipfile.ZipFile(path) as z:
            bad = z.testzip()
        if bad:
            return False, f'ZIP integrity check failed at: {bad}'
    except Exception as exc:
        return False, f'ZIP could not be checked: {exc}'
    return True, 'ZIP structure and CRC checks passed'

valid, reason = check_zip(ARCHIVE)
if valid:
    print('Using the archive already saved in Google Drive — no download needed.')
    print('Archive:', ARCHIVE, '| size:', f'{ARCHIVE.stat().st_size:,}', 'bytes')
else:
    print('No reusable archive found:', reason)
    print('Trying Mendeley automatic download once...')
    api_base = f'https://api.data.mendeley.com/datasets/{DATASET_ID}'
    try:
        meta_response = requests.get(api_base, params={'version': VERSION}, timeout=45)
        if meta_response.status_code in (401, 403):
            raise RuntimeError(f'Mendeley API requires authentication (HTTP {meta_response.status_code}); this notebook does not ask you to paste credentials.')
        meta_response.raise_for_status()
        metadata = meta_response.json()
        print('Dataset:', metadata.get('name'), '| version:', metadata.get('version'))
        zip_response = requests.get(api_base + '/zip', params={'version': VERSION}, timeout=45)
        if zip_response.status_code in (401, 403):
            raise RuntimeError(f'Archive endpoint requires authentication (HTTP {zip_response.status_code}).')
        zip_response.raise_for_status()
        zip_meta = zip_response.json()
        print('Archive status:', zip_meta.get('status'), '| expected bytes:', zip_meta.get('size'))
        for attempt in range(12):
            if zip_meta.get('url') or zip_meta.get('status') in ('COMPLETE', 'COMPLETED', 'READY'):
                break
            time.sleep(10)
            zip_response = requests.get(api_base + '/zip', params={'version': VERSION}, timeout=45)
            zip_response.raise_for_status()
            zip_meta = zip_response.json()
        if not zip_meta.get('url') and zip_meta.get('status') not in ('COMPLETE', 'COMPLETED', 'READY'):
            raise RuntimeError('Mendeley has not prepared the ZIP archive yet.')
        file_response = requests.get(api_base + '/zip/file_downloaded', params={'version': VERSION}, stream=True, timeout=120)
        if file_response.status_code in (401, 403):
            raise RuntimeError(f'Download endpoint requires authentication (HTTP {file_response.status_code}).')
        file_response.raise_for_status()
        h = hashlib.sha256()
        with PARTIAL.open('wb') as out:
            for chunk in file_response.iter_content(4 * 1024 * 1024):
                if chunk:
                    out.write(chunk)
                    h.update(chunk)
        expected_size = zip_meta.get('size')
        if expected_size and PARTIAL.stat().st_size != int(expected_size):
            raise RuntimeError(f'Download size mismatch: expected {expected_size}, got {PARTIAL.stat().st_size}.')
        expected_hash = zip_meta.get('sha256_hash')
        if expected_hash and h.hexdigest().lower() != expected_hash.lower():
            raise RuntimeError('Downloaded ZIP SHA-256 does not match Mendeley metadata.')
        ok, detail = check_zip(PARTIAL)
        if not ok:
            raise RuntimeError(detail)
        PARTIAL.replace(ARCHIVE)
        print('Archive downloaded and saved in Drive. SHA-256:', h.hexdigest())
    except Exception as exc:
        print('\nAutomatic download did not work:', str(exc))
        print('\nThis is a one-time step only:')
        print('1) Open the Mendeley dataset page and tap Download All:')
        print('   https://data.mendeley.com/datasets/8d9fv6kpt3/3')
        print('2) Save the ZIP in Google Drive > MyDrive > DrPlant')
        print('3) Name it exactly: 8d9fv6kpt3-v3.zip')
        print('4) Rerun this cell. Future runs will reuse it automatically.')
        raise RuntimeError('Dataset archive is not available yet. Complete the one-time save to Drive above, then rerun this cell.')

ok, detail = check_zip(ARCHIVE)
if not ok:
    raise RuntimeError(f'The saved dataset archive failed validation: {detail}. Replace the ZIP in Drive before continuing.')
print('Dataset archive ready:', f'{ARCHIVE.stat().st_size:,}', 'bytes —', detail)

In [ ]:
# Extract only when needed; avoid repeating extraction during reruns in the same Colab session.
import zipfile
EXTRACTED = WORK / 'lemon-v3'
image_ext = {'.jpg','.jpeg','.png','.webp','.bmp','.tif','.tiff'}
def is_class_root(path):
    if not path.is_dir(): return False
    dirs = [p for p in path.iterdir() if p.is_dir() and not p.is_symlink()]
    return len(dirs) >= 3 and sum(any(x.suffix.lower() in image_ext for x in d.rglob('*') if x.is_file()) for d in dirs) >= 3

existing = [p for p in [EXTRACTED] + list(EXTRACTED.rglob('*')) if is_class_root(p)] if EXTRACTED.exists() else []
if existing:
    IMAGE_ROOT = min(existing, key=lambda p: len(p.parts))
    print('Reusing extracted dataset:', IMAGE_ROOT)
else:
    EXTRACTED.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ARCHIVE) as z:
        root = EXTRACTED.resolve()
        for info in z.infolist():
            target = (EXTRACTED / info.filename).resolve()
            if root not in target.parents and target != root:
                raise RuntimeError('Unsafe path inside ZIP: ' + info.filename)
        z.extractall(EXTRACTED)
    candidates = [p for p in [EXTRACTED] + list(EXTRACTED.rglob('*')) if is_class_root(p)]
    if not candidates: raise RuntimeError('Could not identify class folders. Inspect extracted archive layout before proceeding.')
    IMAGE_ROOT = min(candidates, key=lambda p: len(p.parts))
    print('Extracted dataset:', IMAGE_ROOT)
print('Class folders:', sorted(p.name for p in IMAGE_ROOT.iterdir() if p.is_dir()))

In [ ]:
# Inventory, SHA-256 duplicate scan, and image decoding check.
AUDIT = WORK / 'audit'
!python scripts/audit_image_classification_dataset.py "$IMAGE_ROOT" --dataset-id 8d9fv6kpt3 --version 3 --license 'CC BY 4.0 (Mendeley V3 record)' --out "$AUDIT" --verify-images
import json
audit_report = json.loads((AUDIT/'archive-audit-report.json').read_text())
print('Images:', audit_report['image_count'])
print('Class counts:', audit_report['class_image_counts'])
print('Exact duplicate groups:', audit_report['exact_duplicate_groups'])
print('Unreadable images:', len(audit_report['unreadable_images']))
if audit_report['image_count'] < 100: raise RuntimeError('Unexpectedly small archive; stop and inspect the dataset before training.')

In [ ]:
# Exploratory split: exact byte-identical files stay together; near-duplicates and site/plant/session leakage are NOT ruled out.
SPLITS = WORK / 'exploratory-splits'
!python scripts/prepare_exploratory_splits.py "$AUDIT/image-manifest.csv" --out "$SPLITS" --seed 42
print((SPLITS/'exploratory-split-report.json').read_text())

In [ ]:
# Train a first MobileNetV3-Small baseline. Test metrics are exploratory, not independent validation.
OUTPUT = WORK / 'model-output'
!python scripts/train_image_classifier.py "$IMAGE_ROOT" "$SPLITS/exploratory-splits.csv" --out "$OUTPUT" --epochs 12 --batch-size 32 --pretrained --exploratory-non-independent
print((OUTPUT/'metrics.json').read_text())

In [ ]:
# Preserve outputs in Drive without deleting earlier runs.
import shutil
DEST = DRIVE / 'lemon-v3-exploratory-baseline'
DEST.mkdir(parents=True, exist_ok=True)
for item in OUTPUT.iterdir():
    target = DEST / item.name
    if item.is_dir():
        if target.exists(): shutil.rmtree(target)
        shutil.copytree(item, target)
    else:
        shutil.copy2(item, target)
for source in (AUDIT/'archive-audit-report.json', SPLITS/'exploratory-split-report.json'):
    shutil.copy2(source, DEST/source.name)
print('Saved to Google Drive:', DEST)
print('Do not integrate this checkpoint into the app until an independent group-aware evaluation is available.')